In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import CLIPModel
import kagglehub


# 1. DATA PIPELINE (This creates train_loader)

# Download the dataset
print("Downloading/Locating dataset...")
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
train_dir = os.path.join(path, 'Training')
test_dir = os.path.join(path, 'Testing')

# Define transforms based on the research paper
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load datasets and create the missing train_loader!
print("Setting up DataLoaders...")
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


# 2. MODEL DEFINITION


class FineTunedCLIPClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super(FineTunedCLIPClassifier, self).__init__()

        # Load the base CLIP model
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        # Freeze the entire text encoder
        for param in self.clip.text_model.parameters():
            param.requires_grad = False

        # Freeze the entire vision encoder initially
        for param in self.clip.vision_model.parameters():
            param.requires_grad = False

        # Unfreeze ONLY the last 2 layers of the vision encoder
        for layer in self.clip.vision_model.encoder.layers[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Custom Classification Head
        self.classification_head = nn.Sequential(
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
        image_embeds = vision_outputs.pooler_output
        logits = self.classification_head(image_embeds)
        return logits


# 3. TRAINING LOOP


print("Initializing model and training parameters...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = FineTunedCLIPClassifier(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()

# AdamW Optimizer
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)

# Cosine Annealing scheduler
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5)

# Start Training
epochs = 30
print("Starting training loop...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # NOW train_loader is defined and ready to go!
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

Downloading/Locating dataset...
Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Setting up DataLoaders...
Initializing model and training parameters...
Training on device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting training loop...
Epoch [1/30] | Loss: 0.3789 | Accuracy: 86.07%
Epoch [2/30] | Loss: 0.2284 | Accuracy: 91.89%
Epoch [3/30] | Loss: 0.1522 | Accuracy: 94.82%
Epoch [4/30] | Loss: 0.1045 | Accuracy: 96.21%
Epoch [5/30] | Loss: 0.0802 | Accuracy: 97.46%
Epoch [6/30] | Loss: 0.1637 | Accuracy: 94.41%
Epoch [7/30] | Loss: 0.1589 | Accuracy: 94.57%
Epoch [8/30] | Loss: 0.1063 | Accuracy: 96.30%
Epoch [9/30] | Loss: 0.0733 | Accuracy: 97.46%
Epoch [10/30] | Loss: 0.0441 | Accuracy: 98.54%
Epoch [11/30] | Loss: 0.1205 | Accuracy: 95.75%
Epoch [12/30] | Loss: 0.1220 | Accuracy: 95.52%
Epoch [13/30] | Loss: 0.0811 | Accuracy: 97.16%
Epoch [14/30] | Loss: 0.0548 | Accuracy: 98.04%
Epoch [15/30] | Loss: 0.0393 | Accuracy: 98.84%
Epoch [16/30] | Loss: 0.1119 | Accuracy: 96.29%
Epoch [17/30] | Loss: 0.0855 | Accuracy: 96.68%
Epoch [18/30] | Loss: 0.0638 | Accuracy: 98.00%
Epoch [19/30] | Loss: 0.0402 | Accuracy: 98.50%
Epoch [20/30] | Loss: 0.0320 | Accuracy: 98.98%
Epoch [21/30] | Loss: 0

In [2]:
import torch.nn as nn
from transformers import BlipModel

class FineTunedBLIPClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super(FineTunedBLIPClassifier, self).__init__()

        # 1. Load the base BLIP model from Salesforce (the creators of BLIP)
        # The paper notes adapting "Salesforce's BLIP-Model"
        self.blip = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base")

        # 2. Freeze the entire text encoder
        for param in self.blip.text_model.parameters():
            param.requires_grad = False

        # 3. Freeze the entire vision encoder initially
        for param in self.blip.vision_model.parameters():
            param.requires_grad = False

        # 4. Unfreeze ONLY the last 2 layers of the vision encoder
        for layer in self.blip.vision_model.encoder.layers[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        # 5. Custom Classification Head (Linear 768 -> 512 -> 4)
        # Identical to the CLIP setup for a fair comparison
        self.classification_head = nn.Sequential(
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        # Extract features using the BLIP vision encoder
        vision_outputs = self.blip.vision_model(pixel_values=pixel_values)

        # BLIP's vision model outputs a sequence, we take the first token (CLS token)
        image_embeds = vision_outputs.last_hidden_state[:, 0, :]

        # Pass through the classification head to get the 4 tumor guesses
        logits = self.classification_head(image_embeds)
        return logits